# Visualize correlations between Hemolysis and SNPs 
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from itertools import chain, combinations, product
import re
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import seaborn as sns

from hk1_r12ter_analysis.config import (
    GENE_ID,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    RAW_DATA_DIR,
    REPORTS_DIR,
    logger,
)
from hk1_r12ter_analysis.io import load_dataframe_from_file, make_filename
from hk1_r12ter_analysis.linkage_disequilibrium import identify_independent_snps
from hk1_r12ter_analysis.util import ceil_to_decimal
from hk1_r12ter_analysis.visualize import plot_hemolysis_pvalues_for_snp

## REDS Index

In [ ]:
sample_key = "INDEX DONOR ID"
group_key = None
source_type = "REDS-Index"
main_data_type = f"Genomics-{GENE_ID}"
other_data_type = "Metadata"

correlation_statistic = "Spearman"
r2_threshold = 0.8
top_n_snps = 30
prioritize_genotyped_in_LD = True

filetype = "pdf"
variable_fancy_replacements = {
    f"{source_type.split('-')[-1]}.Adjusted.Osmotic.Hemolysis": "Osmotic Hemolysis",
    f"{source_type.split('-')[-1]}.Adjusted.Storage.Hemolysis": "Storage Hemolysis",
    f"{source_type.split('-')[-1]}.Adjusted.Oxidative.Hemolysis": "Oxidative Hemolysis",
    source_type: source_type.replace("-", " "),
}

### Load metadata and genomics

In [ ]:
# Load metadata + genomics
input_dirpath = PROCESSED_DATA_DIR / "REDS"
filename = make_filename(source_type, "Donor", f"{main_data_type}_{other_data_type}")
df_metadata_genomics = load_dataframe_from_file(
    input_dirpath / filename, index_col=sample_key, header=[0, 1]
)
df_metadata_genomics = df_metadata_genomics.convert_dtypes(convert_integer=False)
metadata_columns = df_metadata_genomics.loc[:, "Metadata"].columns.tolist()
genomic_columns = df_metadata_genomics.loc[:, f"Genomics-{GENE_ID}"].columns.tolist()
df_metadata_genomics

### Load genomics metadata

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS"
input_filename = make_filename("REDS", "Genomics", GENE_ID, "Metadata")
df_genomics_ann = load_dataframe_from_file(input_dirpath / input_filename, index_col="ID")
df_genomics_ann = df_genomics_ann.loc[genomic_columns]
df_genomics_ann = df_genomics_ann.drop("ANN_Priority", axis=1)
df_rsid_map = df_genomics_ann["RSID"].replace(":", ".")
df_genomics_ann

### Load correlation and statistics

In [ ]:
index_cols = ["Group", "DataType1", "Variable1", "DataType2", "Variable2"]
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
input_filename = make_filename(source_type, f"Genomics-{GENE_ID}_All_Data", "Statistics")

df_correlations_tall = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=None, low_memory=False
)

group_value = "ALL"
correlations_options = {
    "Group": group_value,
    "DataType1": f"Genomics-{GENE_ID}",
    "DataType2": "Metadata",
}
for key, value in correlations_options.items():
    df_correlations_tall = (
        df_correlations_tall[df_correlations_tall[key] == value]
        .drop_duplicates()
        .drop(key, axis=1)
    )
df_correlations_tall = df_correlations_tall.set_index(["Variable1", "Variable2"])
df_correlations_tall.index.names = ["ID", "Variable"]
df_correlations_tall = df_correlations_tall.swaplevel(0, 1)
df_correlations_tall

#### Load linkage disequilibrium $r^{2}$ correlation matrices

In [ ]:
LD_method = "RH"
rsids = df_genomics_ann.index.tolist()
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
input_filename = make_filename(
    "REDS-Index",  # Always REDS-Index cohort
    f"Genomics-{GENE_ID}_Metadata",
    LD_method,
    "LinkageDisequilibrium",
    "ALL",
)
df_ld_matrix = load_dataframe_from_file(input_dirpath / input_filename, index_col=["SNP1", "SNP2"])
df_r2_matrix = df_ld_matrix.loc[:, "r2"]
df_r2_matrix

### Identify top independent SNPs for given variables to display

In [ ]:
# Guaruntee that the top n SNPs for each variable are included for display, order matters
top_n_snps_for_variable_display_guaranteed = {
    f"{source_type.split('-')[-1]}.Adjusted.Osmotic.Hemolysis": 5,
    f"{source_type.split('-')[-1]}.Adjusted.Oxidative.Hemolysis": 5,
    f"{source_type.split('-')[-1]}.Adjusted.Storage.Hemolysis": 5,
}
ordered_variables = list(top_n_snps_for_variable_display_guaranteed)


top_snps_for_variables_dict = {}
for variable in ordered_variables:
    logger.debug(f"Identifying top '{top_n_snps}' independent SNPs for '{variable}'...")
    df_correlations_variable = df_correlations_tall.loc[variable].copy()
    if prioritize_genotyped_in_LD:
        df_correlations_variable.loc[:, "Type"] = (
            df_correlations_variable.reset_index(drop=False)["ID"]
            .str.split("_", n=1, expand=True)[0]
            .values
        )
        df_correlations_variable = df_correlations_variable.sort_values(
            by=["Type", f"{correlation_statistic} -log10(p-value)"], ascending=[True, False]
        )
    else:
        df_correlations_variable = df_correlations_tall.loc[variable].sort_values(
            by=f"{correlation_statistic} -log10(p-value)", ascending=False
        )
    independent_snps = identify_independent_snps(
        df_r2_matrix=df_r2_matrix,
        list_of_snps=df_correlations_variable.index.unique().tolist(),
        r2_threshold=r2_threshold,
    )
    df_correlations_variable = df_correlations_variable.loc[independent_snps].sort_values(
        by=f"{correlation_statistic} -log10(p-value)", ascending=False
    )
    # Top SNPs for variable
    top_snps_for_variables_dict[variable] = df_correlations_variable.index[:top_n_snps].tolist()
    logger.info(f"Identified top '{top_n_snps}' independent SNPs for '{variable}'.")

snps_guaranteed = list(
    set.union(
        *[
            set(top_snps_for_variables_dict[variable][:n_guarunteed])
            for variable, n_guarunteed in top_n_snps_for_variable_display_guaranteed.items()
        ]
    )
)
snps_ordered_unique = list(
    dict.fromkeys(
        list(snps_guaranteed) + list(chain.from_iterable(top_snps_for_variables_dict.values()))
    )
)
logger.debug(f"Number of unique SNPs across variables: '{len(snps_ordered_unique)}'.")
logger.debug(
    f"Number of unique SNPs across variables guaranteed for display: '{len(snps_guaranteed)}'."
)
snps_ordered_unique_independent = identify_independent_snps(
    df_r2_matrix=df_r2_matrix,
    list_of_snps=snps_ordered_unique,
    r2_threshold=r2_threshold,
)
snps_guaranteed = [snp for snp in snps_guaranteed if snp in snps_ordered_unique_independent]
logger.info(
    f"Number of unique independent SNPs across variables guaranteed for display: '{len(snps_guaranteed)}'."
)
snps_to_display = snps_ordered_unique_independent[: max([len(snps_guaranteed), top_n_snps])]
logger.info(f"Number of unique SNPs across variables to display: '{len(snps_to_display)}'.")

# Clean up and format DataFrame for plotting
df_correlations_variables = df_correlations_tall.loc[
    pd.IndexSlice[ordered_variables, snps_to_display], :
]
# Swap index to SNP IDs and replace variable names.
df_correlations_variables = df_correlations_variables.reset_index(drop=False).set_index("ID")
df_correlations_variables["Variable"] = df_correlations_variables["Variable"].replace(
    variable_fancy_replacements
)
# Set variables as categorical data, set order, then sort.
df_correlations_variables["Variable"] = pd.Categorical(df_correlations_variables["Variable"])
df_correlations_variables["Variable"] = df_correlations_variables[
    "Variable"
].cat.reorder_categories(
    [variable_fancy_replacements[variable] for variable in ordered_variables], ordered=True
)
df_correlations_variables = df_correlations_variables.sort_values(
    ["Variable", f"{correlation_statistic} -log10(p-value)"], ascending=[True, False]
)
df_correlations_variables.head(15)

### Visualize sigificance for top independent SNPs in given variables to display

In [ ]:
group_options = {
    "ALL": {"yscalar": 40, "ydecimals": 0, "ymajor_tick_multiple": 40},
}
ylim_pads = (0.025, 0.0)

palette = {
    "Osmotic Hemolysis": "xkcd:cerulean blue",
    "Oxidative Hemolysis": "xkcd:green",
    "Storage Hemolysis": "xkcd:red",
}
color_text_by_variant = {
    "3_prime_UTR_variant": "xkcd:purple",
    "5_prime_UTR_variant": "xkcd:pumpkin",
    "downstream_gene_variant": "xkcd:peacock blue",
    "upstream_gene_variant": "xkcd:deep green",
    "intron_variant": "xkcd:dark brown",
    "synonymous_variant": "xkcd:magenta",
    "missense_variant": "xkcd:red",
    # "missense_variant&splice_region_variant": "black",
    # "splice_acceptor_variant&intron_variant": "black",
    # "splice_region_variant&intron_variant": "black",
    # "splice_region_variant&non_coding_transcript_exon_variant": "black",
}

group_value_kwargs = group_options[group_value]


data = df_correlations_variables.copy()
data.index = pd.Index(
    [df_rsid_map.loc[rsid] if rsid in df_rsid_map else rsid for rsid in data.index],
    name=data.index.name,
)
title = "\n".join(
    [
        f"{GENE_ID} SNP Correlates with Hemolysis",
        f"{variable_fancy_replacements.get(source_type)} Donors"
        + (f", {group_key} {group_value}" if group_value != "ALL" else ""),
    ]
)

yabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} -log10(p-value)"].abs().max(),
    decimals=group_value_kwargs["ydecimals"],
    scalar=group_value_kwargs["yscalar"],
)
ylim = (-yabsmax * ylim_pads[0], yabsmax * (1 + ylim_pads[1]))
print(f"y-limits: {ylim[0]:.4f}, {ylim[1]:.4f}")


fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax = plot_hemolysis_pvalues_for_snp(
    data=data,
    x="ID",
    y=f"{correlation_statistic} -log10(p-value)",
    hue="Variable",
    ax=ax,
    palette=palette,
    xaxis_kwargs=dict(
        # lim=(-0.5, len(data)/3),
        label=f"{GENE_ID} SNP",
        tick_labelsize="medium",
        tick_rotation=60,
        ha="right",
        which="major",
    ),
    yaxis_kwargs=dict(
        lim=(-yabsmax * ylim_pads[0], yabsmax * (1 + ylim_pads[1])),
        label="-log$_{10}$($p$-value)",
        major_tick_multiple=group_value_kwargs["ymajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
        ha="right",
        which="both",
    ),
    grid_kwargs=dict(
        linestyle="-",
        color="xkcd:light grey",
    ),
    fontdict=dict(size="large"),
    legend_kwargs=dict(
        title=None,
        frameon=True,
        edgecolor="black",
        bbox_to_anchor=(0.5 if color_text_by_variant else 0.75, 1),
        loc="upper left",
    ),
)
ax.set_title(title)
ax.hlines(
    y=-np.log10(0.05),
    xmin=ax.get_xlim()[0] - 1,
    xmax=ax.get_xlim()[-1] + 1,
    colors="xkcd:black",
    linestyle="--",
    zorder=5,
)
xtick_int = ax.get_xticks()[1] - ax.get_xticks()[0]
ax.set_xlim(ax.get_xticks()[0] - xtick_int, ax.get_xticks()[-1] + xtick_int)
if color_text_by_variant:
    variants = df_genomics_ann.loc[
        df_rsid_map[df_rsid_map.isin(data.index.unique())].index, "ANN_Annotation"
    ].unique()
    for ticklabel in ax.get_xticklabels():
        rsid = df_rsid_map[df_rsid_map == ticklabel.get_text()].index.item()
        variant = df_genomics_ann.loc[rsid, "ANN_Annotation"]
        color = color_text_by_variant.get(variant, "xkcd:grey")
        ticklabel.set_color(color)

    texts_vbox = mpl.offsetbox.VPacker(
        children=[
            mpl.offsetbox.TextArea(
                text.replace("_", " "),
                textprops=dict(color=color, horizontalalignment="left", weight="bold"),
            )
            for text, color in sorted(
                color_text_by_variant.items(), key=lambda x: len(x[0]), reverse=True
            )
            if text in variants
        ],
        pad=0,
        sep=0,
    )
    ann = mpl.offsetbox.AnnotationBbox(
        texts_vbox,
        (0.765, 0.96 - 0.02 * len(variants)),
        xycoords=ax.transAxes,
        box_alignment=(0, 0.5),
        bboxprops=dict(facecolor="white", edgecolor="black", boxstyle="round"),
    )
    ann.set_figure(fig)
    fig.artists.append(ann)


sns.despine(ax=ax)
output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / main_data_type
    / other_data_type
    / f"{source_type.split('-')[-1]}.Hemolysis"
)
output_dirpath.mkdir(parents=True, exist_ok=True)
filename_args = [
    source_type,
    "Hemolysis",
    f"{correlation_statistic}Correlations",
]

filename_args += [f"top{df_correlations_variables.index.nunique()}"]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
if prioritize_genotyped_in_LD:
    filename_args += ["g_priority"]
output_filename = make_filename(*filename_args, filetype=filetype)
fig.savefig(output_dirpath / output_filename)
output_filename